# Aktywacja Strategii Odbicie Ogórkowe
Ten notatnik służy do testowania, wizualizacji i optymalizacji strategii powrotu do średniej po mocnych spadkach.

## Importy, dane i sygnały

In [1]:
import os
import sys

# Dodajemy folder glowny do path aby moduly dzialaly
sys.path.append(os.path.abspath('c:/Users/PC/Documents/Antigravity/lyse-lby'))
os.chdir('c:/Users/PC/Documents/Antigravity/lyse-lby')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Importujemy ladowanie danych i nasze nowe moduly
from core.ladowanie_danych import create_stock_dfs
from odbicie.mackowe_sygnaly import mackowe_sygnaly
from odbicie.odbicie import generate_odbicie_entries
from odbicie.odbicie_atr import generate_odbicie_atr_entries
from odbicie.odbicie_bb import generate_odbicie_bb_entries
from odbicie.tbm import moving_triple_barrier_labels
from odbicie.optymalizacja import optimize_atr_tbm, optimize_bb_tbm

# Katalog na pliki cache - zawsze wewnatrz odbicie/dane/, niezalezny od cwd
# __vsc_ipynb_file__ dostepny w VS Code; fallback na abspath('')
_SCRIPT_DIR = os.path.dirname(os.path.abspath(globals()["__vsc_ipynb_file__"])) if "__vsc_ipynb_file__" in globals() else os.path.abspath("")
# Jesli notebook lezy wewnatrz odbicie/, katalog jest katalogiem rodzica
if os.path.basename(_SCRIPT_DIR) != 'odbicie':
    _SCRIPT_DIR = os.path.join(_SCRIPT_DIR, 'odbicie')
DANE_DIR = os.path.join(_SCRIPT_DIR, 'dane')
os.makedirs(DANE_DIR, exist_ok=True)
print(f"Cache dir: {DANE_DIR}")

# Ustawienia ladowania danych
settings = {
    'market': 'stocks',
    'interval': '1week',
    'vol_enabled': True,
    'vol_ratio_window': 20,
    'vol_ratio_threshold': 1.2,
    'cmo_enabled': True,
    'cmo_len': 6,
    'cmo_thres': -35,
    'cmo_thres_prev': -50
}

ModuleNotFoundError: No module named 'odbicie.mackowe_sygnaly'

In [ ]:
# 1. Ladowanie Danych
import pickle

data_cache_file = os.path.join(DANE_DIR, 'dfs_cache.pkl')

if os.path.exists(data_cache_file):
    print("Znaleziono zapisane dane. Wczytywanie z pliku...")
    with open(data_cache_file, "rb") as f:
        dfs_1d, dfs_1w = pickle.load(f)
    print(f"Wczytano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W z pliku {data_cache_file}.")
else:
    print("Ladowanie danych dziennych i tygodniowych...")
    dfs_1d, dfs_1w = create_stock_dfs(settings)
    print(f"Zaladowano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W.")
    print("Zapisywanie danych do pliku...")
    with open(data_cache_file, "wb") as f:
        pickle.dump((dfs_1d, dfs_1w), f)
    print("Dane zapisane pomyslnie.")


Znaleziono zapisane dane. Wczytywanie z pliku...
Wczytano 478 symboli 1D i 478 symboli 1W z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\dane\dfs_cache.pkl.


In [ ]:
# 2. Generowanie Sygnalow Bazowych (mackowe_sygnaly)
signals_cache_file = os.path.join(DANE_DIR, 'signals_cache.pkl')

if os.path.exists(signals_cache_file):
    print("Znaleziono zapisane sygnaly. Wczytywanie z pliku...")
    with open(signals_cache_file, "rb") as f:
        signals_df = pickle.load(f)
    print(f"Wczytano {len(signals_df)} sygnalow z pliku {signals_cache_file}.")
else:
    signals_df = mackowe_sygnaly(
        dfs=dfs_1w,
        settings=settings,
        require_vol_confirmation=True,
        require_cmo_confirmation=True,
        interval='1w',
        entry_offset=0,
        pattern_cols=['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line'],
        debug=True
    )
    print("Zapisywanie sygnalow do pliku...")
    with open(signals_cache_file, "wb") as f:
        pickle.dump(signals_df, f)
    print("Sygnaly zapisane pomyslnie.")

signals_df.describe()


Znaleziono zapisane sygnaly. Wczytywanie z pliku...
Wczytano 912 sygnalow z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\dane\signals_cache.pkl.


,signal_time,entry_time,signal_close
count,912,912,912.000000
mean,2023-10-08 09:01:34.736842,2023-10-08 09:01:34.736842,129.103488
min,2021-07-25 00:00:00,2021-07-25 00:00:00,7.970000
25%,2022-05-15 00:00:00,2022-05-15 00:00:00,51.740002
50%,2023-10-08 00:00:00,2023-10-08 00:00:00,97.340000
75%,2025-03-16 00:00:00,2025-03-16 00:00:00,179.127495
max,2026-02-22 00:00:00,2026-02-22 00:00:00,565.369995
std,NaN,NaN,103.256636


## Wejście i Wyjście

In [ ]:
# 3. Wybor Strategii Wejscia
# Zmien ta zmienna aby przełaczyc strategie: 'base', 'atr', 'bb'
STRATEGY = 'base'

if STRATEGY == 'base':
    # --- Strategia bazowa: staly prog procentowy ---
    threshold_pct = 0.11 
    entries_df = generate_odbicie_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        threshold_pct=threshold_pct,
        max_setup_hold_bars=10
    )
    print(f"[base] Wygenerowano {len(entries_df)} wejsc przy progu {threshold_pct*100}%")

elif STRATEGY == 'atr':
    # --- Strategia ATR: prog oparty na wielokrotnosci ATR ---
    atr_period = 20
    atr_factor = 4.0
    entries_df = generate_odbicie_atr_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        atr_period=atr_period,
        atr_factor=atr_factor,
        max_setup_hold_bars=10
    )
    print(f"[atr] Wygenerowano {len(entries_df)} wejsc (period={atr_period}, factor={atr_factor})")

elif STRATEGY == 'bb':
    # --- Strategia BB: wejscie przy dotknięciu dolnej wstegi Bollingera ---
    bb_period = 20
    bb_std   = 2.0
    entries_df = generate_odbicie_bb_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        bb_period=bb_period,
        bb_std=bb_std,
        max_setup_hold_bars=10
    )
    print(f"[bb] Wygenerowano {len(entries_df)} wejsc (period={bb_period}, std={bb_std})")

else:
    raise ValueError(f"Nieznana strategia: '{STRATEGY}'. Uzyj 'base', 'atr' lub 'bb'.")

entries_df.head()


[base] Wygenerowano 107 wejsc przy progu 11.0%


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr
0,ABNB,2022-06-19,hammer,2022-06-30,88.546098,99.489998,0.11,6.897889
1,ALB,2023-11-05,hammer,2023-11-09,113.902203,127.980003,0.11,7.207769
2,ALB,2023-10-29,inverted_hammer,2023-11-01,119.651602,134.440002,0.11,7.000488
3,ARE,2025-11-23,hammer,2025-12-08,45.292099,50.889999,0.11,2.208598
4,AXP,2025-03-23,engulfing_bull,2025-04-04,237.919998,270.510010,0.11,10.113220


In [ ]:
# Params           Więcej:                                                              Mniej:
tpm = 1.25    #    - łapiemy większe ruchy (ryzykujemy powrotem).                        - ratujemy i szybciej zbieramy mniejsze kwoty.
slm = 2.5     #    - luźniejszy stop loss (wytrzymuje szum korekcyjny).                  - szybsza kapitulacja i ucinanie straty z palca.
ttpm = 0.25   #    - luźniejsze spuszczanie kursu w trendach, nie zostajemy wyrzuceni.   - szybsze zabezpieczanie małego peaku.
mhb = 15      #    - dajemy kapitałowi długo leżeć pod ruchem bocznym.                   - szukamy szybkich obrotów uwalnaijąc portfel.

# Nowe parametry wyjścia czasowego i ochrony kapitału
early_bailout = False     # Ucieczka w połowie czasu (mhb/2) jeśli trade jest na minusie
time_decay_sl = False     # Stop loss podnosi się z czasem w kierunku ceny wejścia
active_trail_sl = False   # Stop loss podąża za każdym nowym szczytem, nie tylko po aktywacji TP
sl_trail_mult = 3.0       # Z jakiej odległości (w ATR) ma podążać aktywny SL
max_loss_pct = 0.15       # Twardy cap straty (15%). Chroni przed ogromnymi ATR-ami na groszówkach.

In [ ]:
# Params V => overfitted, ale z głową
tpm = 1.2
slm = 2.7
ttpm = 0.1
mhb = 100               # disabled
early_bailout = False   # disabled
time_decay_sl = True
active_trail_sl = True 
sl_trail_mult = 3.0
max_loss_pct = 1        # disabled
exit_on_close=True


In [ ]:
# 4. Wyjście z użyciem Moving Triple Barrier Method
trades_df = moving_triple_barrier_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=tpm,
    sl_mult=slm,
    tp_trail_mult=ttpm,
    max_holding_bars=mhb,
    early_breakeven=early_bailout,
    time_decay_sl=time_decay_sl,
    active_trailing_sl=active_trail_sl,
    sl_trail_mult=sl_trail_mult,
    max_loss_pct=max_loss_pct,
    exit_on_close=exit_on_close
)
print(f"Zakończono {len(trades_df)} transakcji.")
trades_df.head()


Zakończono 107 transakcji.


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr,exit_time,exit_price,return_pct,exit_reason,hold_bars
0,ABNB,2022-06-19,hammer,2022-06-30,88.546098,99.489998,0.11,6.897889,2022-07-11,95.099998,7.401682,TRAILING_TP,6
1,ALB,2023-11-05,hammer,2023-11-09,113.902203,127.980003,0.11,7.207769,2023-11-16,122.599998,7.636196,TRAILING_TP,5
2,ALB,2023-10-29,inverted_hammer,2023-11-01,119.651602,134.440002,0.11,7.000488,2023-11-06,119.459999,-0.160134,TRAILING_TP,3
3,ARE,2025-11-23,hammer,2025-12-08,45.292099,50.889999,0.11,2.208598,2025-12-19,47.939999,5.846272,TRAILING_TP,9
4,AXP,2025-03-23,engulfing_bull,2025-04-04,237.919998,270.510010,0.11,10.113220,2025-04-10,246.889999,3.770175,TRAILING_TP,4


## Analiza

In [ ]:
# 5. Analiza i Statystyki
if not trades_df.empty:
    wins = (trades_df['return_pct'] > 0).sum()
    losses = (trades_df['return_pct'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    trades_df['return_per_bar'] = trades_df['return_pct'] / trades_df['hold_bars']
    
    print(f"Total Trades: {len(trades_df)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg Return: {trades_df['return_pct'].mean():.2f}%")
    print(f"Avg bars held: {trades_df['hold_bars'].mean():.2f}")
    print(f"Avg Return per Bar: {trades_df['return_per_bar'].mean():.2f}%")

    
    # Powody wyjścia
    print("\nExit Reasons:")
    print(trades_df['exit_reason'].value_counts())
else:
    print("Brak transakcji do analizy.")

Total Trades: 107
Win Rate: 80.37%
Avg Return: 6.11%
Avg bars held: 11.02
Avg Return per Bar: 1.04%

Exit Reasons:
exit_reason
TRAILING_TP    92
TRAILING_SL    14
SL              1
Name: count, dtype: int64


In [ ]:
temp = pd.DataFrame({
    'count': trades_df.groupby('exit_reason').return_pct.count(),
    'avg_return': trades_df.groupby('exit_reason').return_pct.mean(),
    'cumulativ_return': trades_df.groupby('exit_reason').return_pct.sum(),
    'std': trades_df.groupby('exit_reason').return_pct.std(),
    'avg_hold_bars': trades_df.groupby('exit_reason').hold_bars.mean(),
    'std_hold_bars': trades_df.groupby('exit_reason').hold_bars.std(),
    'max_hold_bars': trades_df.groupby('exit_reason').hold_bars.max()
    })
    
temp

,count,avg_return,cumulativ_return,std,avg_hold_bars,std_hold_bars,max_hold_bars
exit_reason,,,,,,,
SL,1,-13.406540,-13.406540,NaN,1.000000,NaN,1
TRAILING_SL,14,-15.482631,-216.756829,6.359662,22.000000,12.347532,44
TRAILING_TP,92,9.609483,884.072395,9.095764,9.456522,10.541819,54


## Ploty

In [ ]:
# Ploty — wyswietla interaktywny widz transakcji

module_path = r"d:\Antigravity\rebound\lyse-lby\odbicie"
if module_path not in sys.path:
    sys.path.append(module_path)

from odbicie.plot import show_trade_viewer

show_trade_viewer(
    trades_df,
    dfs_1d,
    tpm, slm, ttpm, mhb,
    exit_reason='all',
    active_trailing_sl=active_trail_sl,
    sl_trail_mult=sl_trail_mult,
    max_loss_pct=max_loss_pct,
    time_decay_sl=time_decay_sl,
    exit_on_close=exit_on_close,
    strategy_type='tbm',
)


Output()

## Optymalizacja

### Stare

In [ ]:
# Optymalizacja Progu Wejścia i Czasu Trzymania Setupu
import itertools
from tqdm.notebook import tqdm

def optimize_threshold(thresholds, max_holding_bars):
    results = []
    
    # Tworzymy siatkę wszystkich kombinacji wejściowych list
    grid = list(itertools.product(thresholds, max_holding_bars))
    
    for th, max_bars in tqdm(grid, desc="Optymalizacja progu"):
        # max_bars definiuje ile dni po sygnale czekamy na wpadnięcie w próg
        ents = generate_odbicie_entries(signals_df, dfs_1d, threshold_pct=th, max_setup_hold_bars=max_bars)
        
        # max_bars definiuje również jak długo trzymamy trade zanim zamkniemy na czas
        trds = moving_triple_barrier_labels(ents, dfs_1d, tp_mult=tpm, sl_mult=slm, tp_trail_mult=ttpm, max_holding_bars=max_bars)
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            results.append({
                'threshold_pct': th,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return
            })
            
    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='avg_return', ascending=False)
    return df_res

'''
thresholds = [9,10,11,12]
max_holding_bars = [15]

print("Uruchamianie optymalizacji progu...")
opt_df = optimize_threshold(thresholds, max_holding_bars)

display(opt_df.head(10))
'''


'\nthresholds = [9,10,11,12]\nmax_holding_bars = [15]\n\nprint("Uruchamianie optymalizacji progu...")\nopt_df = optimize_threshold(thresholds, max_holding_bars)\n\ndisplay(opt_df.head(10))\n'

In [ ]:
# Optymalizacja Parametrów TBM (Take Profit / Stop Loss / Max Hold)
import itertools
from tqdm.notebook import tqdm
import pandas as pd

def optimize_tbm(entries_df, market_data_daily, tp_mults, sl_mults, trail_activations, max_holding_bars_list):
    results = []
    
    # Tworzymy siatkę wszystkich kombinacji
    grid = list(itertools.product(tp_mults, sl_mults, trail_activations, max_holding_bars_list))
    
    for tp, sl, trail, max_bars in tqdm(grid, desc="Optymalizacja TBM"):
        trds = moving_triple_barrier_labels(
            entries_df=entries_df, 
            market_data_daily=market_data_daily, 
            tp_mult=tp, 
            sl_mult=sl, 
            tp_trail_mult=trail, 
            max_holding_bars=max_bars,
            early_breakeven=early_bailout,
            time_decay_sl=time_decay_sl,
            active_trailing_sl=active_trail_sl,
            sl_trail_mult=sl_trail_mult,
            max_loss_pct=max_loss_pct
        )
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            avg_hold_bars = trds['hold_bars'].mean()

            results.append({
                'tp_mult': tp,
                'sl_mult': sl,
                'trail_activation': trail,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return,
                'avg_hold_bars': avg_hold_bars,
                'return_per_bar': avg_return / avg_hold_bars if avg_hold_bars > 0 else 0
            })

    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='return_per_bar', ascending=False)
    return df_res


'''
tp_mults = [a/100 for a in range(113, 117, 1)]
sl_mults = [a/100 for a in range(313, 321, 1)]
trail_activations = [a/100 for a in range(1, 3, 1)]
max_holding_bars = [15]

print("Uruchamianie optymalizacji TBM. To może zająć chwilę...")
tbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)

display(tbm_opt_results.head(10))
'''


'\ntp_mults = [a/100 for a in range(113, 117, 1)]\nsl_mults = [a/100 for a in range(313, 321, 1)]\ntrail_activations = [a/100 for a in range(1, 3, 1)]\nmax_holding_bars = [15]\n\nprint("Uruchamianie optymalizacji TBM. To może zająć chwilę...")\ntbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)\n\ndisplay(tbm_opt_results.head(10))\n'

### ATR + TBM

In [ ]:
# Optymalizacja wejsc ATR + wyjsc TBM
# Odkomentuj blok ponizej, aby uruchomic optymalizacje.

'''
atr_opt_results = optimize_atr_tbm(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    atr_periods=[14, 20, 30],
    atr_factors=[2.0, 3.0, 4.0],
    max_setup_hold_bars_list=[10, 15],
    tp_mults=[1.0, 1.25, 1.5],
    sl_mults=[2.0, 2.5, 3.0],
    tp_trail_mults=[0.1, 0.25],
    max_holding_bars_list=[15, 30],
    exit_on_close=False,
    min_trades=10
)
display(atr_opt_results.head(20))
'''

'\natr_opt_results = optimize_atr_tbm(\n    signals_df=signals_df,\n    market_data_daily=dfs_1d,\n    atr_periods=[14, 20, 30],\n    atr_factors=[2.0, 3.0, 4.0],\n    max_setup_hold_bars_list=[10, 15],\n    tp_mults=[1.0, 1.25, 1.5],\n    sl_mults=[2.0, 2.5, 3.0],\n    tp_trail_mults=[0.1, 0.25],\n    max_holding_bars_list=[15, 30],\n    exit_on_close=False,\n    min_trades=10\n)\ndisplay(atr_opt_results.head(20))\n'

### BB + TBM

In [ ]:
# Optymalizacja wejsc BB + wyjsc TBM
# Odkomentuj blok ponizej, aby uruchomic optymalizacje.

'''
bb_opt_results = optimize_bb_tbm(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    bb_periods=[15, 20, 30],
    bb_stds=[1.5, 2.0, 2.5],
    max_setup_hold_bars_list=[10, 15],
    tp_mults=[1.0, 1.25, 1.5],
    sl_mults=[2.0, 2.5, 3.0],
    tp_trail_mults=[0.1, 0.25],
    max_holding_bars_list=[15, 30],
    exit_on_close=False,
    min_trades=10
)
display(bb_opt_results.head(20))
'''

'\nbb_opt_results = optimize_bb_tbm(\n    signals_df=signals_df,\n    market_data_daily=dfs_1d,\n    bb_periods=[15, 20, 30],\n    bb_stds=[1.5, 2.0, 2.5],\n    max_setup_hold_bars_list=[10, 15],\n    tp_mults=[1.0, 1.25, 1.5],\n    sl_mults=[2.0, 2.5, 3.0],\n    tp_trail_mults=[0.1, 0.25],\n    max_holding_bars_list=[15, 30],\n    exit_on_close=False,\n    min_trades=10\n)\ndisplay(bb_opt_results.head(20))\n'